# 02 — Tariff lists mapped to food CPI series: treated, placebo and control series

Next step 3 in `CONTRIBUTING.md` (decision D18). Sources:
- 2018: schedule of SOR/2018-153 [R10], repealed by SOR/2019-144 [R9]; amendment SOR/2019-111 removed boats only [R11].
- 2025 and 2026: Department of Finance complete list, current and previous versions [R7].
- Tariff rates within vs over access commitment: CBSA Customs Tariff [S8, S9].

Versioned inputs: `data/reference/tariff_hs6_food.csv` (food HS chapters 01–24) and `data/reference/tariff_cpi_map.csv` (HS → CPI, with exposure type and confidence).
Logic: `src/tariff_map.py`. Output: `data/processed/treatment_roles.csv` (not versioned).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else "src")
import pandas as pd
import tariff_map as T
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 80)

## 1. Food tariff lines per episode

In [2]:
hs6 = pd.read_csv(T.HS6_FILE, dtype=str)
hs6["n_lines"] = hs6.n_lines.astype(int)
hs6["quota"] = hs6.access.map(lambda a: "over-access only" if a == "O" else ("no quota" if a == "N" else "within-access included"))
summary = hs6.pivot_table(index="event_id", columns="quota", values="n_lines", aggfunc="sum", fill_value=0)
summary["total_lines"] = summary.sum(axis=1)
summary["hs6_subheadings"] = hs6.groupby("event_id").size()
summary

quota,no quota,over-access only,within-access included,total_lines,hs6_subheadings
event_id,,,,,
CA2018,23,0,2,25,18
CA2025,228,54,125,407,183
CA2026,5,0,61,66,15


In [3]:
# Out of CPI food scope: alcohol (22.03-22.08), tobacco (24), live animals (01) and animal feed (23)
out = hs6[hs6.hs6.str[:4].between("2203", "2208") | hs6.chapter.isin(["01", "23", "24"])]
out.groupby("event_id").n_lines.sum().rename("lines_out_of_food_cpi")

event_id
CA2018      1
CA2025    119
Name: lines_out_of_food_cpi, dtype: int64

## 2. Mapping to CPI series

In [4]:
tmap = T.read_map()
pd.crosstab(tmap.event_id, tmap.exposure, margins=True)

exposure,direct,input,over_quota_only,partial,All
event_id,,,,,
CA2018,10,0,0,4,14
CA2025,23,2,5,8,38
CA2026,5,5,0,1,11
All,38,7,5,13,63


## 3. Roles per episode (series with enough history, D17)

In [5]:
r = T.roles()
out = T.ROOT / "data" / "processed" / "treatment_roles.csv"
out.parent.mkdir(parents=True, exist_ok=True)
r.to_csv(out, index=False)
pd.crosstab(r[r.pre_ok].event_id, r[r.pre_ok].role, margins=True)

role,control,excluded_aggregate,excluded_input,excluded_other_episode,excluded_partial,out_of_scope,placebo,treated,All
event_id,,,,,,,,,
CA2018,71,17,0,0,4,4,0,10,106
CA2025,30,26,2,0,8,4,11,25,106
CA2026,39,8,7,50,1,4,0,5,114
All,140,51,9,50,13,12,11,40,326


In [6]:
conf = tmap.set_index(["event_id", "cpi_product"]).confidence
treated = r[r.role == "treated"].copy()
treated["confidence"] = [conf.get((e, v)) for e, v in zip(treated.event_id, treated.mapped_via)]
treated[["event_id", "product", "mapped_via", "confidence", "pre_months", "post_months", "usable"]]

,event_id,product,mapped_via,confidence,pre_months,post_months,usable
51,CA2018,Other dairy products,Other dairy products,medium,402,98,True
59,CA2018,Other bakery products,Other bakery products,medium,402,98,True
77,CA2018,Fruit juices,Fruit juices,high,402,98,True
80,CA2018,"Jam, jelly and other preserves (2013=100)","Jam, jelly and other preserves (2013=100)",high,66,98,True
101,CA2018,Confectionery,Confectionery,high,402,98,True
107,CA2018,Roasted or ground coffee (2013=100),Roasted or ground coffee (2013=100),high,66,98,True
111,CA2018,Fermented or pickled vegetables (2013=100),Fermented or pickled vegetables (2013=100),high,66,35,True
112,CA2018,"Sauces, condiments and dips (2013=100)","Sauces, condiments and dips (2013=100)",high,66,35,True
115,CA2018,Soup,Soup,high,478,98,True
124,CA2018,Non-alcoholic beverages,Non-alcoholic beverages,medium,690,98,True


## 4. Placebo series (2025, over-access lines only)

These supply-managed products were on the 2025 list, but only for imports above the quota, which already pay MFN rates of about 240–300% [S9]. The added 25% should change nothing: a treated-like effect on them would point to a problem in the method (falsification test).

In [7]:
r[(r.role == "placebo") & r.pre_ok][["event_id", "product", "mapped_via", "usable"]]

,event_id,product,mapped_via,usable
170,CA2025,Fresh milk,Fresh milk,True
171,CA2025,Whole milk (2013=100),Fresh milk,True
172,CA2025,Low-fat milk (2013=100),Fresh milk,True
173,CA2025,Butter,Butter,True
174,CA2025,Cheese,Cheese,True
175,CA2025,Cheddar cheese (2013=100),Cheese,True
176,CA2025,Processed cheese (2013=100),Cheese,True
177,CA2025,Unripened or fresh cheese (2013=100),Cheese,True
178,CA2025,Mozzarella cheese (2013=100),Cheese,True
179,CA2025,Ice cream and related products,Ice cream and related products,True


## 5. Controls for 2026

The 2026 pre-period (36 months, from 2023-09) contains the 2025 episode, so series exposed in 2025 are excluded (`excluded_other_episode`). Placebo series stay eligible.

In [8]:
r[(r.event_id == "CA2026") & (r.role == "control") & r.pre_ok][["product", "depth", "is_leaf", "pre_months"]]

,product,depth,is_leaf,pre_months
261,Fresh or frozen meat (excluding poultry),3,False,576
262,Fresh or frozen beef,4,False,668
270,Fresh or frozen pork,4,False,576
274,Other fresh or frozen meat (excluding poultry),4,True,500
284,Ham and bacon,4,False,500
285,Bacon (2013=100),5,True,164
286,Ham excluding deli ham (2013=100),5,True,164
289,Fish,3,False,576
290,Fresh or frozen fish (including portions and fish sticks),4,True,500
291,Canned and other preserved fish,4,False,576
